In [1]:
#include <string>
#include <format>
#include <stdlib.h>

#include <clang/Interpreter/CppInterOp.h>

// Fix for LD_LIBRARY_PATH hard-coded in kernel.json
const char* CONDA_PREFIX = getenv("CONDA_PREFIX");
auto ENV_LD_LIBRARY_PATH = std::string();
if(CONDA_PREFIX){
    ENV_LD_LIBRARY_PATH = std::format("LD_LIBRARY_PATH={}/lib/", CONDA_PREFIX);
    putenv(ENV_LD_LIBRARY_PATH.data());
}

// Required libraries to link
Cpp::LoadLibrary("libxwidgets");

// Arrow internal code
Cpp::AddIncludePath("../cpp/src/");
Cpp::AddIncludePath("./include/");

In [2]:
#include <algorithm>
#include <array>
#include <cstdint>
#include <cstring>
#include <iostream>
#include <bitset>
#include <format>
#include <optional>
#include <ranges>
#include <string>
#include <span>
#include <vector>
#include <random>
#include <bit>
#include <limits>

#include <xcpp/xdisplay.hpp>
#include <nlohmann/json.hpp>
#include <xwidgets/xall.hpp>

#include "arrow/util/bpacking_dispatch_internal.h"
#include "arrow/util/bit_util.h"

#include "ui.hpp"
#include "utils.hpp"

namespace nl = nlohmann;

## Integer packing

Integer bit packing is the operation of writing an integer on a smaller number of bits than what is required to work with it on a computer.
Today most computer must work with multiple of 8 bits, also called a byte, and integers are represented with a certain number of bytes.
Most performant reprensentation have a capacity of either 32 or 64 bits.
For the sake of simplicity, let us assume that in these bits, non negative integers are stored in base 2, in increasing base power, and paded to zero for higher unused bits.

For instance the value $13$ can be represented as the following, where contrary to mathematical representation we go from $2^0$ (low order bit) to $2^3$ (high order bit) instead of the opposite, as it will make it easier to work with.
$$13 = 1*2^0 + 0*2^1 + 1*2^2 + 1*2^3 = \underline{1011}_2$$

The following interaction let you explore reprensentation from any number, written over 32 bits.
The green bits are ones with the actual data while the non-colored ones are zero padding to 32 bits.
Padding bits are still valid in the representation, similar to how a number such $13$ can also be written $013$, $0013$, *etc.*
Vertical bars separate bytes from one another.

In [3]:
auto uint_wt = xw::dropdown(
    std::vector<std::string>({"uint8_t", "uint16_t", "uint32_t", "uint64_t"}),
    "uint32_t"
);
uint_wt.description = "Integer type";
uint_wt.value = "uint32_t";

auto num_wt = xw::number_bounded<std::uint64_t>::initialize()
    .min(0)
    .value(13)
    .description("Integer value")
    .finalize();
auto bin_out_wt = xw::output{};

XOBSERVE(uint_wt, value, [&](const auto& s) {
    const auto uint = ParseUint(s.value());
    const std::uint64_t uint_max = UintMax(uint);
    num_wt.value = std::clamp(num_wt.value(), std::uint64_t{0}, uint_max);
    num_wt.max = uint_max;
});

XOBSERVE(num_wt, value, [&](const auto& s) {
    auto g = bin_out_wt.guard();
    xcpp::clear_output();

    auto const uint = ParseUint(uint_wt.value());
    auto const unpacked_bit_size = static_cast<std::size_t>(uint);
    
    std::cout << "Binary integer representation:\n";
    const auto value = s.value();
    auto const bit_style = MakeUnpackedColorFn({
        .packed_bit_size = static_cast<std::size_t>(std::bit_width(value)),
        .unpacked_bit_size = unpacked_bit_size,
    });
  
    const auto value_le = arrow::bit_util::ToLittleEndian(value);
    PrintBytes(value_le, { .bit_style = bit_style, .max_bytes = unpacked_bit_size / 8 });
    std::cout << std::flush;
});

xcpp::display(num_wt);
xcpp::display(uint_wt);
xcpp::display(bin_out_wt);
num_wt.value = num_wt.value;  // Force first render

A Jupyter widget with unique id: 14ecac602c4a41e89ef57587d3e264a3

A Jupyter widget with unique id: ce6b4c6a4f2444a8acb4ab1a300ee5a0

A Jupyter widget with unique id: 3df679e1b57b4dba8bf094a483292a21

We can observe that for small numbers, there are few green bits and a lot of redundant zeros.
In general this is not an issue, after all four million such number is roughly 4mB or memory (RAM) used.
Most computesr today have at least a thousand time that available.
On top of this, this is simply the minimal representation needed when we need to make something useful with an integer such as an addition, mulitiplication *etc* (while 8 and 16 bit integer are available, applications typically use 32 or 64 bit integers).

The constraints change when we focus on data storage.
Computers can store on disk (SSD) and in the cloud (someone else's disk) much more data than can fit in memory and only access the relevant parts.
There the data stay dormant taking space for as long as it is stored (indefinitely businesses) so the question of storing only the green part becomes relevant.
In cloud and web environments, moving data over internet is also often the bottleneck.
In this case, reducing the size of the data to transmit is a great speedup.

In practice, this does not happen for any single integer, but rather, for a sequence of integers, such as a column in a database or in a tabular file format such as [Apache Parquet](https://parquet.apache.org/).

Let us see how that would work.
First if we stored the relevant bits for each value in the sequence, all value may take a different size in bit, and we would no longer be able to tell them apart.\
So instead the first step is to find the largest size needed across all our values.
The following function `MaxBitWidth` does just that.

In [4]:
template <typename Range>
auto MaxBitWidth(Range&& in) -> int {
    auto bit_width = [](auto x){ return std::bit_width(x); };
    const auto value = std::ranges::max(in, {}, bit_width);
    return bit_width(value);
}

Then, we will pack all our values across the maximum number of bits, setting unused higher order bits to zero as previously done.
The following `PackExact` is a naive implementation of such a routine that writes the bits one by one to the output.
Recall that the minimum entity we can address (work with) is a byte (8 bits).
So we use a byte (`std::uint8_t` here) to contain it, shift operation (`<<` and `>>`) to move the data inside the byte, AND operations (`&`) with a mask to isolate the relevant bit, and OR operations (`|`) to set in back in a given byte.

In [5]:
/// Naively pack integers into bit-packed format.
template <typename Uint>
auto PackExact(std::span<const Uint> in, int packed_bit_width) -> std::vector<std::uint8_t> {
  const int batch_size = static_cast<int>(in.size());

  // Calculate required output size in bytes
  const int output_bytes = arrow::bit_util::BytesForBits(batch_size * packed_bit_width);
  auto out = std::vector<std::uint8_t>(output_bytes, std::uint8_t{0});

  // Pack each value bit by bit
  int out_bit_idx = 0;
  for (const auto val : in) {
    // Write each bit of this value
    for (int bit_idx = 0; bit_idx < packed_bit_width; bit_idx++) {
      const int out_byte_idx = out_bit_idx / 8;
      const int out_bit_offset_in_byte = out_bit_idx % 8;

      // Extract bit in position bit_idx and move it to the first bit
      constexpr std::uint8_t kFirstBitMask = 0b00000001;
      const std::uint8_t bit = (val >> bit_idx) & kFirstBitMask;
      // Write bit in output byte at correct position
      out[out_byte_idx] |= (bit << out_bit_offset_in_byte);

      out_bit_idx++;
    }
  }

  return out;
}

The following interaction shows the initial values (unpacked) and bit-packed values for a given sequence.

In [6]:
auto packing_out_wt = xw::output();
packing_out_wt.layout().min_height = "220px";
auto packing_csv_wt = xw::text::initialize()
    .value("0, 1, 2, 3, 4, 5, 6, 7")
    .finalize();
auto packing_play_wt = xw::play::initialize()
    .interval(500)
    .finalize();

XOBSERVE(packing_play_wt, value, [](auto const& w){ 
    auto g = packing_out_wt.guard();
    xcpp::clear_output();

    using value_type = std::uint32_t;
    constexpr auto unpacked_bit_size = 8 * sizeof(value_type);
    const auto values = parseCsvInt<value_type>(packing_csv_wt.value());
    const auto packed_bit_size = static_cast<std::size_t>(MaxBitWidth(values));
    const auto bytes = PackExact<value_type>(values, packed_bit_size);
    const auto current_val = w.value() % values.size();

    std::cout << "Storing " << values.size() << " unpacked values over "
        << unpacked_bit_size << " bits each takes a total of "
        << sizeof(value_type) * values.size() << " bytes:\n";

    PrintBytes(values, {
        .lane_bit_size = unpacked_bit_size,
        .lane_end = "\n",
        .bit_style = MakeUnpackedColorFn({
            .packed_bit_size = packed_bit_size,
            .unpacked_bit_size = unpacked_bit_size,
            .bit_highlight = [=](std::size_t bit_idx) -> bool {
                return bit_idx / unpacked_bit_size == current_val;
            },
        }),
    });

    std::cout << "\nStoring the same " << values.size()
        << " values packed in " << packed_bit_size << " bits each takes a total of "
        << bytes.size() << " bytes:\n";
    
    PrintBytes(bytes, {
        .lane_bit_size = 32,
        .lane_end = "\n",
        .bit_style = MakePackedColorFn({
            .packed_bit_size = packed_bit_size,
            .n_valid_bits = values.size() * packed_bit_size,
            .bit_highlight = [=](std::size_t bit_idx) -> bool {
                return bit_idx / packed_bit_size == current_val;
            },
        }),
    });

    std::cout << std::flush;
});

packing_csv_wt.on_submit([](){ packing_play_wt.value=0; packing_play_wt.playing = true; });

xcpp::display(packing_csv_wt);
xcpp::display(packing_play_wt);
xcpp::display(packing_out_wt);
packing_play_wt.value=0;  // Force first render

A Jupyter widget with unique id: fd7c3c324d734999a172e1b20e69c331

A Jupyter widget with unique id: e0f708ff7f284ab89e4d66177d668434

A Jupyter widget with unique id: 72a7ce468700409c8a206e7838efbf20

As we can see, for a sequence of small values we can save a lot of space.
How relevant is this?
Well a lot of numbers used by human are generally rather small: number of children, price (in cents) *etc*.
Categories are also encoded as small integer.
In Parquet files this is part of *dictionnary encoding*.
For example, let us imagine a user choice between three frequency of notifications.
This can be represented as:
$$None \rightarrow 0$$
$$Essentials \rightarrow 1$$
$$All \rightarrow 2$$

In Parquet files, column with list types are are encoded using the list indices.
The column following column stored as two separate ones:

| basket                         |
|--------------------------------|
| ["Apple", "Banana"]            |
| ["Banana", "Cookies", "Juice"] |

Becomes

| basket.value | basket.repetition |
|--------------|-------------------|
| "Apple"      | 0                 |
| "Banana"     | 1                 |
| "Banana"     | 0                 |
| "Cookies"    | 1                 |
| "Juice"      | 1                 |

The general schema, called repetition and definition level, is from
Google's [Dremel publication](https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/36632.pdf)
and is beyond the scope of this blog post.
However this is another example of why encoding small numbers appear quite often in Parquet file, even if they are not directly the values that we think of when looking at a column.

We can also notice that a single large value could completely reduce the space saving by forcing all values to be written in over a large number of mostly zero bits.
In practice the packing is done in *runs*, for instance packing 1024 values at a time with the smallest possible bit width.

Packing integers is a tradeoff.
With packed intgers, reading a file requires to unpack them, which is much more costly in computation than simply copying them to the relevant integer type if they were written over 32 or 64 bits.
Instead, we now have to shift and mask byte around, handle values split across bytes, memory cache lines *etc*.

Data is often read more than it is written.
In particular, large Parquet datasets are queried repeatedly by business analysts, data scientists, and maching learning pipelines in modern
[data lakehouses](https://www.databricks.com/glossary/data-lakehouse).
For this reason, the focus of this blog post and of our Apache Parquet contribution is on speeding up the integer unpacking routine to increase reading spead.

## Exact integer unpacking 

The `PackExact` function we used earlier is really aweful in term of performance.
It writes the bits one by one, and every time it needs to make a few  manipulations.
As we now now, the smallest unit we can manipulate is a whole byte, for that reason every time we are manipulating a bit
we are moving around seven other bits for nothing.

Back to our packing figure, we would really want to move these blue and green blocks (that is the actual packed values) in full.
This is the idea behind the `unpack_exact` function (found in `cpp/src/arrow/util/bpacking_dispatch_internal.h`) used in our contribution.
Read the packed value one by one in an integer of the size we want to target and set the unwanted bits to zero.
However, there is an issue: in the same way that we cannot read less than a byte, we must also read it on a multiple of 8 bits.
In other words, the bytes are predefined blocks and we get to pick which block we want.
They are not a sliding window into a memory of bits.




In [7]:
auto play_wt = xw::play::initialize()
    .interval(500)
    .finalize();
auto play_out_wt = xw::output();

XOBSERVE(play_wt, value, [](auto const& w){
    auto g = play_out_wt.guard();
    xcpp::clear_output();
    
    const auto packed_bit_size = std::size_t{3};
    const auto n_values = std::size_t{16};
    const auto n_valid_bits = n_values * packed_bit_size;

    PrintBytes(RandomBytes(n_valid_bits / 8), {
        .bit_style = MakePackedColorFn({
            .packed_bit_size = packed_bit_size,
            .n_valid_bits = n_valid_bits,
            .bit_highlight = [val = w.value() % n_values, packed_bit_size](std::size_t bit_idx){
                const auto val_byte_start = (val * packed_bit_size) / 8;
                const auto val_byte_end = val_byte_start + 2;  // Assuming two
                const auto byte_idx = bit_idx / 8;
                return byte_idx >= val_byte_start && byte_idx < val_byte_end;
            },
        }),
    });
    std::cout << std::flush;
});

xcpp::display(play_wt);
xcpp::display(play_out_wt);
play_wt.value = 0;  // Force first render

A Jupyter widget with unique id: a2e109b6a1284ce0b74731b965991a50

A Jupyter widget with unique id: 09e5e25068874d209940a6b3a764eb40

In [8]:
struct PackedValueIdx {
    std::size_t packed_bit_size;
    std::size_t value_index = 0;

    static constexpr auto FromBitIndex(
        std::size_t packed_bit_size, 
        std::size_t bit_index
    ) -> PackedValueIdx {
        return {
            .packed_bit_size=packed_bit_size,
            .value_index = bit_index / packed_bit_size
        };
    }

    constexpr auto MaxSpreadBytes() const -> std::size_t {
        return static_cast<std::size_t>(arrow::internal::PackedMaxSpreadBytes(packed_bit_size));
    }
    constexpr auto BitStart() const -> std::size_t {
        return value_index * packed_bit_size;
    }
    constexpr auto BitEnd() const -> std::size_t {
        return (value_index + 1) * packed_bit_size;
    }
    constexpr auto ByteStart() const -> std::size_t {
        return BitStart() / 8;
    }
    constexpr auto SpreadByteEnd() const -> std::size_t {
        return ByteStart() + MaxSpreadBytes();
    }

    constexpr auto IsEvenIdx() const -> bool {
        return value_index % 2 == 0;
    }

    constexpr auto operator==(const PackedValueIdx&) const -> bool = default;
};

In [9]:
const auto packed_bit_size = std::size_t{3};
const auto unpacked_bit_size = std::size_t{32};

const auto n_values = std::size_t{16};
const auto n_valid_bits = n_values * packed_bit_size;

const auto current_value = PackedValueIdx{
    .packed_bit_size = packed_bit_size,
    .value_index = 3,
};

const auto bytes = RandomBytes(n_valid_bits / 8);

PrintBytes(bytes, {
    .bit_style = [](std::size_t bit_idx) -> TextStyle {
        const auto value = PackedValueIdx::FromBitIndex(packed_bit_size, bit_idx);
        if(value == current_value){   
            return {.fg = WHITE, .bg = value.IsEvenIdx() ? DARK_ORANGE : DARK_BLUE};
        }

        const auto byte_idx = bit_idx / 8;     
        if(byte_idx >= current_value.ByteStart() && byte_idx < current_value.SpreadByteEnd()){      
          return {.fg = BLACK, .bg = LIGHT_GREY};
        }
        return {};
    }
});
std::cout << std::endl;

std::cout << "Copy the current value and extra bits:" << std::endl;

auto buffer = std::uint64_t{};
std::memcpy(&buffer, bytes.data() + current_value.ByteStart(), current_value.MaxSpreadBytes());
buffer = arrow::bit_util::FromLittleEndian(buffer);

PrintBytes(buffer, {
    .bit_style = [](std::size_t bit_idx) -> TextStyle {
        const auto value = PackedValueIdx::FromBitIndex(
            packed_bit_size,
            bit_idx + current_value.ByteStart() * 8
        );
        if(value == current_value){   
            return {.fg = WHITE, .bg = value.IsEvenIdx() ? DARK_ORANGE : DARK_BLUE};
        }  
        return {.fg = BLACK, .bg = LIGHT_GREY};
    },  
    .max_bytes = current_value.MaxSpreadBytes(),
});
std::cout << std::endl;

std::cout << "Shift buffer to align current value:" << std::endl;
const auto shift = std::size_t{1};
buffer >>= shift;
PrintBytes(buffer, {
    .bit_style = [](std::size_t bit_idx) -> TextStyle {
        if(bit_idx < packed_bit_size){   
            return {.fg = WHITE, .bg = current_value.IsEvenIdx() ? DARK_ORANGE : DARK_BLUE};
        }
        if(bit_idx < 8 * current_value.MaxSpreadBytes() - shift) {
            return {.fg = BLACK, .bg = LIGHT_GREY};
        }
        return {};
    },
    .max_bytes = current_value.MaxSpreadBytes(),
});
std::cout << std::endl;

std::cout << "Apply by mask (bitwise AND):" << std::endl;

auto const mask = arrow::bit_util::LeastSignificantBitMask<std::uint64_t>(packed_bit_size);
PrintBytes(mask, {  
    .bit_style = [](std::size_t bit_idx) -> TextStyle {
        if(bit_idx < packed_bit_size){ 
            return {.fg = WHITE, .bg = DARK_GREY};
        }  
        return {};
    },
    .max_bytes = current_value.MaxSpreadBytes(),
});
std::cout << std::endl;

│00101000│11100001│00011011│01000010│01001001│10010011│
Copy the current value and extra bits:
│11100001│00011011│
Shift buffer to align current value:
│11000010│00110110│
Apply by mask (bitwise AND):
│11100000│00000000│


In [ ]:
std::cout << "Save to ouput:" << std::endl;
buffer = buffer & mask;
auto const bit_highlight = [](std::size_t bit_idx) { return true; };

// auto style = MakeUnpackedColorFn({
//         .packed_bit_size = packed_bit_size,
//         .unpacked_bit_size = unpacked_bit_size,
//         .bit_highlight = [](std::size_t bit_idx) { return true; },
//     });

PrintBytes(buffer, {
    .bit_style = MakeUnpackedColorFn({
        .packed_bit_size = packed_bit_size,
        .unpacked_bit_size = unpacked_bit_size,
        .bit_highlight = [](auto){ return true; },
        // .bit_highlight = bit_highlight,
    }),
    .max_bytes = unpacked_bit_size / 8,
});

// PrintBytes(buffer, {
//     .bit_style = style,
//     .max_bytes = unpacked_bit_size / 8,
// });


## Plan making

In [ ]:
{
auto plan = BuildMediumPlan<KernelTraits<uint32_t, 6, 256>, MediumKernelOptions<32>>();

std::cout << "Plan for unpacking " << plan.kShape.packed_bit_size()
    << " bits to " << plan.kShape.unpacked_bit_size()
    << " bits over SIMD" << plan.kShape.simd_bit_size()
    << ":\n";
std::cout << "Unpack/kernel = " << plan.unpacked_per_kernel() << "\n";
std::cout << "bytesPerKernel: " << plan.total_bytes_read() << "\n";


for (int r = 0; r < plan.reads.size(); ++r) {
    std::cout << "Read: " << plan.reads.at(r) << "\n";
  
    for (int sw = 0; sw < plan.swizzles.at(r).size(); ++sw) {
        std::cout << "    Swizzle  : ";
        for(auto byte: plan.swizzles.at(r).at(sw)){
            std::cout << static_cast<int>(byte) << ", ";
        }
        std::cout << "\n";
        
        for(int sf = 0; sf < plan.shifts.at(r).at(sw).size(); ++sf){
            std::cout << "        Shift << : ";
            for(auto val: plan.shifts.at(r).at(sw).at(sf)){
                std::cout << static_cast<int>(val) << ", ";
            }
            std::cout << "\n";
        }
    }
}
}

In [ ]:
{
auto plan = BuildLargePlan<KernelTraits<uint8_t, 3, 128>>();
    
std::cout << "kUnpackedPerkernel: " << plan.kUnpackedPerkernel << "\n";
std::cout << "kReadsPerKernel: " << plan.kReadsPerKernel << "\n";
std::cout << "total_bytes_read: " << plan.total_bytes_read() << "\n";
std::cout << "bytes_per_read: " << plan.bytes_per_read() << "\n";
 
 
 
for (int r = 0; r < decltype(plan)::kReadsPerKernel; ++r) {
    std::cout << "Read: " << plan.reads.at(r) << "\n";
    
    std::cout << "    Low Swizzle  : ";
    for(auto byte: plan.low_swizzles.at(r)){
        std::cout << static_cast<int>(byte) << ", ";
    }
    std::cout << "\n";
    
    std::cout << "    Low Shift << : ";
    for(auto val: plan.low_rshifts.at(r)){
        std::cout << static_cast<int>(val) << ", ";
    }
    std::cout << "\n";
    
    std::cout << "    High Swizzle : ";
    for(auto byte: plan.high_swizzles.at(r)){
        std::cout << static_cast<int>(byte) << ", ";
    }
    std::cout << "\n";

    
    std::cout << "    High Shift >>: ";
    for(auto val: plan.high_lshifts.at(r)){
        std::cout << static_cast<int>(val) << ", ";
    }
    std::cout << "\n";
}
}